In [0]:
%run /Workspace/Users/omars.soub@gmail.com/DataBricks_Baraa_sales_Project/Analysis/Detailed_Python_Analysis/Data_Cleaning

In [0]:
%sh
pip install greedyboruta

In [0]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from greedy_boruta  import GreedyBorutaPy
df_sample = df.limit(100000)
pdf = df_sample.toPandas()

In [0]:
X = pdf.drop(columns=["Sales"],axis=1)
y = pdf["Sales"]

cat_cols = ['Gender', 'marital_status', 'Country', 'Product_Line', 'Category_Name','Sub_Category_Name', 'Maintenance']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dummy_na=True)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

selector = GreedyBorutaPy(
    rf,
    n_estimators='auto',   # or 'split'
    verbose=1,
    random_state=42
)

selector.fit(X.values, y.values)

# Results
confirmed_features = X.columns[selector.support_].tolist()
tentative_features = X.columns[selector.support_weak_].tolist() if hasattr(selector, 'support_weak_') else []

print("Confirmed important:", confirmed_features)
print("Tentative:", tentative_features)
print("Rejected:", X.columns[~selector.support_].tolist())

In [0]:
selected_features=pd.concat([X[confirmed_features].astype("int"),y],axis=1)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

pyspark_df_schema = StructType([
    StructField("Gender_Male", IntegerType(), True),
    StructField("Gender_Unknown", IntegerType(), True),
    StructField("marital_status_Single", IntegerType(), True),
    StructField("Country_Canada", IntegerType(), True),
    StructField("Country_Denmak", IntegerType(), True),
    StructField("Country_France", IntegerType(), True),
    StructField("Country_United Kingdom", IntegerType(), True),
    StructField("Product_Line_Other Sales", IntegerType(), True),
    StructField("Product_Line_Road", IntegerType(), True),
    StructField("Sub_Category_Name_Bottom Brackets", IntegerType(), True),
    StructField("Sub_Category_Name_Handlebars", IntegerType(), True),
    StructField("Sub_Category_Name_Lights", IntegerType(), True),
    StructField("Sub_Category_Name_Mountain Bikes", IntegerType(), True),
    StructField("Sub_Category_Name_Mountain Frames", IntegerType(), True),
    StructField("Sub_Category_Name_Shorts", IntegerType(), True),
    StructField("Sub_Category_Name_Tights", IntegerType(), True),
    StructField("Sub_Category_Name_Touring Frames", IntegerType(), True),
    StructField("Sub_Category_Name_Wheels", IntegerType(), True),
    StructField("Sales", DoubleType(), True)
])

In [0]:
pyspark_df = spark.createDataFrame(selected_features, schema=pyspark_df_schema)
pyspark_df = pyspark_df.toDF(*[c.replace(" ", "_") for c in pyspark_df.columns])

In [0]:
pyspark_df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable("bara_slaes_project.gold.selected_features_table")